[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/02_pd_orientation.ipynb)

# R2-Q1 Week 1 — Orient on PlantDoc

This notebook orients you on the PlantDoc dataset: 27 classes (17 disease and 10 healthy) across 13 plant species, captured under field conditions that contrast with PlantVillage's controlled lab setup. Together with `01_pv_orientation.ipynb`, the inspection here supports your Week 1 EQ#1 report and the written pre-commits on aggregation level, class-space remapping, and statistical test that R2-Q1 requires before any training begins.

By the end of this notebook you will have:

- A PlantDoc metadata table (`pd_metadata.parquet`) keyed by image path, with class label and host species per image, ready for the transfer evaluation in Week 3.
- Summary statistics on the PlantDoc class space — total image count, per-class distribution, host-species coverage, disease-vs-healthy class split — saved as `pd_class_summary.parquet`.
- A `pd_orientation_summary.json` recording dataset path, image and class counts, host-species count, and key distributional facts, ready as Week 1 EQ#1 input.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R2-Q1 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load the PlantDoc metadata

PlantDoc (Singh et al. 2020) is the second of the two datasets you'll
use in R2-Q1. Where PlantVillage is roughly 54,000 lab-condition leaf
images shot against uniform backgrounds, PlantDoc is roughly 2,500
*field-condition* images — photos of diseased and healthy leaves in
real-world settings, heterogeneous in resolution, lighting, and
background. That heterogeneity is exactly the point: PlantDoc is the
canonical test bed for measuring whether a PV-trained classifier
transfers to the messy conditions a real deployment would see.

The function below returns two things, not one — a `(metadata_df,
hf_dataset)` tuple. The DataFrame is your handle on what's in the
dataset; the `hf_dataset` is a Hugging Face Dataset object that holds
the image bytes themselves. Both are needed because the PyTorch wrapper
you'll use next week (`PlantDocDataset`) takes both: the DataFrame tells
it which rows to iterate over, the Dataset object holds the actual
images.

The first time you call this in a fresh session, expect a 30-to-90-
second pause while the ~950 MB dataset gets fetched from the Hugging
Face Hub. Subsequent calls in the same session are instant.

In [ ]:
### 2.1) Load PlantDoc data
metadata, hf_dataset = iri.load_plantdoc()

print(f"Metadata shape:  {metadata.shape[0]:,} rows × {metadata.shape[1]} columns")
print(f"Columns:         {list(metadata.columns)}")
print(f"HF Dataset:      {len(hf_dataset):,} rows")
print()
print("First five rows:")
metadata.head()

Seven columns of metadata, all string or simple types. Notice what's
*not* here: there's no `image_path` column. PlantDoc's images don't
live as files in a directory you can point to — they live inside the
`hf_dataset` object as PIL Images, indexed by pandas row position.

Two columns carry the same information in different forms. `class_label`
is the upstream folder name verbatim from Singh et al. 2020 ("Apple
Scab Leaf", "Tomato leaf bacterial spot", etc.); `class_idx` is an
integer (0 to 27) assigned by sorting class labels alphabetically with
case sensitivity. The integer form is what your classifier will
predict.

The remaining five columns:

- `host` — normalized host name (e.g., "Apple", "Tomato"). Hand-curated
  from the irregular upstream folder names.
- `disease` — lowercased disease name, or `"healthy"` for healthy leaves.
- `is_healthy` — boolean shorthand for `disease == "healthy"`.
- `split` — `"train"` or `"test"`, from the upstream partition.
- `filename` — original filename verbatim, including any URL-encoded
  characters and double extensions.

**Looking up an image.** Each row's pandas index is the position to use
in `hf_dataset`. To get the image for the first row:

    image = hf_dataset[0]["image"]    # a PIL Image

You'll see this pattern again in Section 6 when you display sample
images.

## 3) Inspect class structure

PlantDoc has 28 classes — 27 disease-on-host classes plus one orphan
you need to know about. Unlike PlantVillage's tidy `host___disease`
naming convention, PlantDoc's class names are inconsistent in
capitalization, word order, and where the word "leaf" appears. The
`host` and `disease` columns are a hand-curated normalization layer
that smooths this out; the `class_label` column preserves the upstream
string verbatim because the link back to Singh et al. 2020 should be
exact.

Start by looking at one row in detail.

In [ ]:
### 3.1) Walk through one row
example = metadata.iloc[0]
print(f"class_label:  {example['class_label']}")
print(f"class_idx:    {example['class_idx']}")
print(f"host:         {example['host']}")
print(f"disease:      {example['disease']}")
print(f"is_healthy:   {example['is_healthy']}")
print(f"split:        {example['split']}")
print(f"filename:     {example['filename']}")

The `class_label` string is what the upstream PlantDoc repository
named the folder this image came from. A few examples of how
inconsistent these strings are:

- `Apple Scab Leaf` — Title Case, disease before "Leaf"
- `Tomato Septoria leaf spot` — mixed case, "leaf spot" as a compound
- `grape leaf` — all lowercase
- `Tomato leaf bacterial spot` — host first, then "leaf", then disease
- `Soyabean` — upstream misspelling of "Soybean", preserved verbatim

This kind of irregularity is normal for web-scraped data; the
discipline of preserving upstream strings is what lets you trace any
individual image back to Singh et al. 2020 unambiguously. Use the
`host` and `disease` columns for any analysis grouping — those are
normalized.

One class is unusual enough to surface explicitly. The class
"Tomato two spotted spider mites leaf" has 2 images in the training
split and 0 images in the test split — a real artifact of the upstream
dataset, not a bug. Singh et al. 2020 and downstream benchmarks like
Ahmad et al. 2023 drop this class implicitly when they report "27
classes." This dataset preserves it for upstream fidelity. You'll need
to decide whether to drop it from your own evaluation; that choice
becomes part of your EQ#1 pre-commit in Section 8.

In [ ]:
### 3.2) Count classes and per-class images
class_counts = metadata['class_label'].value_counts()

print(f"Number of classes:  {len(class_counts)}")
print()
print(f"Largest class:      {class_counts.idxmax()}")
print(f"                    {class_counts.max():,} images")
print(f"Smallest class:     {class_counts.idxmin()}")
print(f"                    {class_counts.min():,} images")
print(f"Median count:       {int(class_counts.median()):,} images per class")
print(f"Ratio largest:smallest:  {class_counts.max() / class_counts.min():.1f}×")
print()

# Surface the orphan class explicitly: which classes have 0 test images?
split_counts = metadata.groupby(['class_label', 'split']).size().unstack(fill_value=0)
no_test = split_counts[split_counts.get('test', 0) == 0]
print("Classes with 0 test images:")
print(no_test if len(no_test) > 0 else "  (none)")

Two things to take in here.

The ratio between largest and smallest classes is well above the noise
floor — PlantDoc isn't balanced. When you train next week on
PlantVillage and evaluate on PlantDoc, the per-class accuracy you'd
compute on the smallest PD classes will come from very few test images
(typically 8 to 12 per class, sometimes fewer). This is the noise
problem the R2-Q1 Considerations section flagged: per-class accuracy
on PD test sets is too noisy to support reliable claims at the per-
class level.

The aggregation question this raises is one of the three pre-commits
you'll write in Section 8. The decision isn't whether to aggregate
(you essentially have to) but at what level — host group, disease
category, or some other grouping you can defend.

The orphan class you saw flagged above is also a per-class-count issue,
just an extreme one. Zero test images means you can't measure your
classifier's accuracy on that class no matter what. Dropping it from
the evaluation is the standard move; preserving it in the metadata for
upstream fidelity is what this dataset does.

In [ ]:
### 3.3) Plot per-class image counts by split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
split_counts_sorted = (
    split_counts
    .assign(total=lambda d: d[["train", "test"]].sum(axis=1))
    .sort_values("total", ascending=False)   # highest totals at top
)

# Convert from wide to long format for seaborn
plot_df = (
    split_counts_sorted[["train", "test"]]
    .reset_index()
    .melt(
        id_vars=split_counts_sorted.index.name or "index",
        value_vars=["train", "test"],
        var_name="split",
        value_name="count"
    )
)

# Rename class column cleanly
plot_df = plot_df.rename(columns={split_counts_sorted.index.name or "index": "class"})

# Preserve descending class order
class_order = split_counts_sorted.index.tolist()

# -----------------------------
# Pastel-ish colors
# -----------------------------
palette = {
    "train": "#9ecae1",  # soft blue
    "test": "#fdae6b",   # soft orange
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 10))

left_values = pd.Series(0, index=class_order)

for split in ["train", "test"]:
    split_df = (
        plot_df[plot_df["split"] == split]
        .set_index("class")
        .loc[class_order]
    )

    ax.barh(
        y=class_order,
        width=split_df["count"],
        left=left_values,
        label=split,
        color=palette[split],
        alpha=0.85
    )

    left_values += split_df["count"]

ax.set_xlabel("Number of images")
ax.set_ylabel("")
ax.set_title("Images per class in PlantDoc: train + test")

# Put highest values at the top
ax.invert_yaxis()

# Cleaner legend placement
ax.legend(
    title="Split",
    loc="center right",
    bbox_to_anchor=(0.98, 0.82),
    frameon=True
)

sns.despine(left=True, bottom=False)

plt.tight_layout()
plt.show()

## 4) Inspect host species

"Host" here means the plant species the leaf came from. Aggregating
from classes to hosts collapses across diseases for the same plant —
every disease of tomato, plus healthy tomato, becomes one "Tomato"
group. This is one candidate aggregation level for your transfer
evaluation: rather than asking "how does my classifier do on tomato
leaf bacterial spot?" you ask "how does it do on tomatoes overall?"

In [ ]:
### 4.1) Count hosts and images per host
host_counts = metadata['host'].value_counts()

print(f"Number of host species:  {len(host_counts)}")
print()
print("Images per host:")
print(host_counts.to_string())

Thirteen distinct hosts in PlantDoc, versus PlantVillage's fourteen.
Tomato is the most-represented host in both datasets, but the absolute
counts differ by an order of magnitude — PlantDoc's per-host counts
are in the dozens-to-hundreds range, not the thousands-to-tens-of-
thousands range.

Two upstream conventions are worth noting because they'll matter when
you build your PV→PD remapping in Section 8:

- "Corn" in PlantDoc corresponds to "Corn_(maize)" in PlantVillage —
  same plant, different upstream label string.
- "Soyabean" in PlantDoc is the upstream misspelling of "Soybean",
  which is how PlantVillage spells it.

If you're aligning host names across the two datasets, you'll need to
handle these by hand. Whichever direction you go (rename one to match
the other, or use a translation dict in your remapping), document it
explicitly — a reader checking your work shouldn't have to guess.

In [ ]:
### 4.2) Plot images per host
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
plot_df = (
    host_counts
    .sort_values(ascending=False)
    .reset_index()
)

plot_df.columns = ["host_species", "count"]

# -----------------------------
# Categorize counts for PlantDoc scale
# -----------------------------
def count_category(n):
    if n > 1000:
        return "> 1000"
    elif n > 500:
        return "501–1000"
    elif n > 250:
        return "251–500"
    else:
        return "≤ 250"

plot_df["count_group"] = plot_df["count"].apply(count_category)

group_order = ["> 1000", "501–1000", "251–500", "≤ 250"]

palette = {
    "> 1000": "#9ecae1",     # soft blue
    "501–1000": "#a1d99b",  # soft green
    "251–500": "#fdae6b",   # soft orange
    "≤ 250": "#fbb4b9",     # soft pink
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=plot_df,
    x="host_species",
    y="count",
    hue="count_group",
    hue_order=group_order,
    palette=palette,
    dodge=False,
    order=plot_df["host_species"],
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Number of images")
ax.set_title("Images per host species in PlantDoc")

plt.xticks(rotation=45, ha="right")

ax.legend(
    title="Image count",
    loc="upper right",
    frameon=True
)

sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

## 5) Inspect disease types

The `disease` column carries a different cut through the same data.
Where the class label tells you "tomato with bacterial spot" and the
host column tells you "tomato," the disease column tells you "bacterial
spot" regardless of host.

Two PD-specific conventions to know about. First, disease names are
*lowercased* in the `disease` column even when the upstream
`class_label` had Title Case (`Apple Scab Leaf` becomes
`disease = "scab"`). This differs from how PlantVillage's loader
handles disease names — be deliberate about case-insensitive matching
if you're comparing across datasets. Second, the disease "two spotted
spider mites" is technically a pest, not a pathological disease; it
shows up in PlantDoc because the upstream folder named it as a class.

Start with the simplest cut: healthy versus diseased.

In [ ]:
### 5.1) Healthy vs diseased
n_healthy = metadata['is_healthy'].sum()
n_diseased = (~metadata['is_healthy']).sum()
print(f"Healthy images:   {n_healthy:,}  ({n_healthy / len(metadata):.1%})")
print(f"Diseased images:  {n_diseased:,}  ({n_diseased / len(metadata):.1%})")

Now look at how many distinct disease labels there are, and what they
are.

In [ ]:
### 5.2) Count and list distinct diseases
unique_diseases = sorted(metadata['disease'].unique())

print(f"Number of distinct disease labels:  {len(unique_diseases)}")
print()
print("All disease labels:")
for d in unique_diseases:
    print(f"  {d}")

The R2-Q1 Considerations section flagged "fungal, bacterial, or viral
disease categories" as one of the candidate aggregation levels. To use
that grouping, you need a mapping from each disease label to its broad
category.

PlantDoc doesn't provide this mapping. The cell below provides a
starter mapping covering the disease labels in the dataset, based on
common plant-pathology references. **Treat it as a draft, not as
authoritative.** Each entry is a verifiable claim ("scab on apple is
fungal") that you should check against your reading list before
locking it in.

Two PD-specific subtleties. First, this mapping uses *lowercased* keys
because that's how PD's `disease` column reads — if you build a
parallel mapping for PlantVillage later, you'll need to handle case
carefully. Second, the "two spotted spider mites" entry is a pest,
not a disease in the pathological sense — same edge case PlantVillage
had with the same pest, and the same choice you'll need to make about
how to group or drop it.

In [ ]:
### 5.3) Apply a starter category mapping
# Starter mapping from PD disease label to broad category.
# Categories: fungal, bacterial, viral, pest, healthy.
# Verify each entry against your reading list before using this in
# your EQ#1 pre-commit. Disease names below match what
# `iri.load_plantdoc()` produces in the `disease` column (lowercased).
pd_disease_category = {
    "scab":                      "fungal",
    "rust":                      "fungal",
    "black rot":                 "fungal",
    "early blight":              "fungal",
    "late blight":               "fungal",
    "leaf blight":               "fungal",
    "gray leaf spot":            "fungal",
    "septoria leaf spot":        "fungal",
    "powdery mildew":            "fungal",
    "leaf mold":                 "fungal",
    "mold":                      "fungal",
    "bacterial spot":            "bacterial",
    "mosaic virus":              "viral",
    "yellow leaf curl virus":    "viral",
    "two spotted spider mites":  "pest",
    "healthy":                   "healthy",
}

# Apply the mapping. Any disease label not in the dict shows up as
# "UNMAPPED" — if you see that, your mapping is incomplete and you
# need to extend it before relying on it for evaluation.
pd_categorized = metadata['disease'].map(pd_disease_category).fillna("UNMAPPED")

print("Images per category:")
print(pd_categorized.value_counts().to_string())
print()
unmapped_count = (pd_categorized == "UNMAPPED").sum()
if unmapped_count > 0:
    unmapped_diseases = metadata.loc[pd_categorized == "UNMAPPED", 'disease'].unique()
    print(f"WARNING: {unmapped_count} images have unmapped disease labels:")
    for d in unmapped_diseases:
        print(f"  {d}")

The category counts are imbalanced in much the same direction as
PlantVillage's: fungal diseases dominate, bacterial and viral are
minority categories. PlantDoc's "pest" category is the same single-
disease edge case PV has — spider mites on tomato.

Two things this surfaces for your evaluation work.

Aggregating to the three-category level (fungal / bacterial / viral)
gives you a small number of categories with workable image counts per
category. This is the "broad disease category" aggregation level the
R2-Q1 page mentions.

The category counts here will be the *upper bound* on your test-set
size at this aggregation level once you've matched against
PlantVillage's class space (Section 7 sets up that comparison; Section
8 commits to a remapping).

## 6) Look at sample images

You've inspected the metadata. Now look at the actual images, because
the visual difference between PlantDoc and PlantVillage is what makes
the lab-to-field gap exist in the first place. PlantVillage was shot
in a studio; PlantDoc was scraped from the open web. Below, four hosts
side by side, healthy and diseased, displayed *at their original
resolution* — no resizing before display. The heterogeneity is the
point.

The image display uses the lookup pattern from Section 2: each row's
pandas index is the position to read from `hf_dataset`. The defensive
`.convert("RGB")` call (which `PlantDocDataset` does for you when you
use it later) is reproduced here because PlantDoc includes at least
one CMYK image, and silently passing CMYK to matplotlib is the kind of
bug that's hard to debug if it happens far from where you can see it.

In [ ]:
### 6.1) Display sample images at original resolution
# Four hosts chosen to overlap with the PV orientation notebook's
# choices. Each contributes one healthy and one diseased sample where
# available. Image titles include the actual resolution to make the
# heterogeneity visible.
sample_hosts = ['Apple', 'Grape', 'Potato', 'Tomato']

samples = []
for host in sample_hosts:
    for state, mask in [('healthy', metadata['is_healthy']),
                        ('diseased', ~metadata['is_healthy'])]:
        host_rows = metadata[(metadata['host'] == host) & mask]
        if len(host_rows) == 0:
            # PlantDoc doesn't have every host in both states.
            # Print a note instead of silently skipping.
            print(f"No {state} samples for host {host} — skipping.")
            continue
        chosen = host_rows.sample(n=1, random_state=42).iloc[0]
        # Use the pandas row index to look up the image in hf_dataset.
        image = hf_dataset[int(chosen.name)]["image"].convert("RGB")
        samples.append((chosen, image))

# Display in a 2-column grid, one row per host-state combination.
# Each axis is sized generously so the native aspect ratio is visible.
n = len(samples)
fig, axes = plt.subplots(n // 2, 2, figsize=(12, 5 * (n // 2)))
for ax, (row, image) in zip(axes.flat, samples):
    ax.imshow(image)
    w, h = image.size
    state_label = "healthy" if row['is_healthy'] else "diseased"
    ax.set_title(
        f"{row['host']} — {state_label} ({row['disease']})\n{w}×{h} pixels",
        fontsize=10
    )
    ax.axis('off')
plt.tight_layout()
plt.show()

A few things to take in.

**Backgrounds vary.** Some images are shot against soil, some against
sky, some against other foliage, some on a human hand or a table.
There is no convention. A model trained on PlantVillage's uniform
gray backdrop has nothing in its training history that helps it
recognize "this is a leaf against soil" as the same kind of input as
"this is a leaf against gray cardboard."

**Lighting varies.** Some images are overexposed, some are in deep
shade, some have dappled light, some have direct flash. Compare this
to PV's controlled studio lighting where exposure and shadow are
roughly the same image to image.

**Resolution varies.** The image titles above show wildly different
pixel counts — width ranges from under 200 pixels to over 4,000 pixels
across the full dataset, and aspect ratios are anything but uniform.
When you train your classifier next week, you'll resize all PD images
to whatever input resolution your CNN expects. That resize is itself a
transformation PV images never had to undergo — they were already
roughly uniform — and it's worth thinking about whether the resize is
doing something to the high-resolution PD images that the low-
resolution PV images escape.

**Capture devices vary.** PV used a small number of cameras under
controlled conditions; PD images came from whatever phone, camera, or
microscope the original photographer used. JPEG compression artifacts,
sensor noise patterns, and color profiles all differ across images.
That's part of what your classifier has to handle in transfer.

None of this is a problem for PlantDoc as a dataset — it's exactly
what makes it useful as a *field-condition* reference. The PV → PD gap
exists because a model trained only on uniform-condition photographs
has no signal in its training history for how to ignore the visual
variation that PD introduces. Sections 7 and 8 set up the methodology
pre-commits you'll need before measuring that gap rigorously.

## 7) PV vs PD: what aligns, what doesn't

You've now seen PlantDoc on its own terms. The transfer question R2-Q1
asks (does a PV-trained classifier work on PD images?) requires you to
align the two datasets' class spaces before you can compute anything
meaningful. PV has 38 classes; PD has 28; the *intersection by class
name* is essentially zero because the two datasets name their classes
differently.

What you need to find is the overlap at *coarser* levels — which
hosts appear in both, which diseases appear in both, which host-
disease combinations appear in both. Section 8 turns this overlap
into a concrete class-space remapping you commit to before any
training happens.

Load PV's metadata so you can compare side by side. If you ran Notebook
01 in this session (or a prior session with the cache mounted), the PV
metadata is already cached and the call below is near-instant. If not,
expect a 30-to-60-second download.

In [ ]:
### 7.1) Load PV metadata for comparison
pv_metadata, pv_hf_dataset = iri.load_plantvillage()
print(f"PV metadata:  {len(pv_metadata):,} rows, {pv_metadata['class_label'].nunique()} classes")
print(f"PD metadata:  {len(metadata):,} rows, {metadata['class_label'].nunique()} classes")

In [ ]:
### 7.2) Compare host overlap
pv_hosts = set(pv_metadata['host'].unique())
pd_hosts = set(metadata['host'].unique())

print("PV-only hosts:")
for h in sorted(pv_hosts - pd_hosts):
    print(f"  {h}")
print()
print("PD-only hosts:")
for h in sorted(pd_hosts - pv_hosts):
    print(f"  {h}")
print()
print("Hosts in both (by exact string match):")
for h in sorted(pv_hosts & pd_hosts):
    print(f"  {h}")

Look at the "Hosts in both" list first. Those are the hosts where a
direct PV → PD transfer evaluation makes sense without any host-level
remapping. Hosts in only one dataset are the ones you'll have to
either drop from your evaluation or remap via a translation step
(e.g., does PV's "Corn_(maize)" count as the same host as PD's
"Corn"? Yes for any sensible analysis, but you have to say so in
writing).

This is the first concrete version of the remapping problem. The PV →
PD direct overlap is partial, and you have to commit to how you'll
handle the partial overlap before you train.

In [ ]:
### 7.3) Compare disease overlap (case-insensitive)
# Lowercase both for fair comparison — PV preserves Title Case, PD
# lowercases. The comparison should look at the underlying disease
# regardless of how each dataset capitalized it.
pv_diseases = set(pv_metadata['disease'].str.lower().unique())
pd_diseases = set(metadata['disease'].str.lower().unique())

print(f"PV diseases:  {len(pv_diseases)}")
print(f"PD diseases:  {len(pd_diseases)}")
print()
print("Diseases in both (lowercased, exact string match):")
for d in sorted(pv_diseases & pd_diseases):
    print(f"  {d}")
print()
print("PV-only diseases:")
for d in sorted(pv_diseases - pd_diseases):
    print(f"  {d}")
print()
print("PD-only diseases:")
for d in sorted(pd_diseases - pv_diseases):
    print(f"  {d}")

The exact-match overlap on disease names is smaller than you might
expect. Several diseases have *almost* the same name across the two
datasets but not exactly — even with lowercase normalization, you'll
see PV's "northern leaf blight" and PD's "leaf blight" as distinct
strings when they're plausibly the same disease.

This is the second concrete version of the remapping problem. The
disease-name fields in the two datasets are normalized differently
enough that string equality isn't a reliable test of "same disease."
You'll need to make explicit judgment calls — does PV's "Cedar apple
rust" map to PD's "rust"? does PV's "Leaf scorch" map to anything in
PD? — and write those calls into your EQ#1 pre-commit.

## 8) EQ#1 pre-commit deliverable

The R2-Q1 question page asks you to commit to three methodology
choices in writing before you start training next week. The reason
the pre-commits exist is to prevent the "garden of forking paths"
problem — if you pick your aggregation level, your remapping, or your
statistical test *after* seeing your results, you'll end up picking
the option that tells the story you want, and your transfer claim
stops being a real test.

Your EQ#1 deliverable is a `precommit.json` file capturing the three
decisions. The cells below walk through each decision with the data
you've seen in this notebook, then provide a template for the JSON.

### 8.1) Decision 1 — Aggregation level

When you measure your classifier's accuracy on PD next week, at what
grouping do you report it?

The candidates you've seen evidence for in this notebook:

- **Per-class** (28 PD classes). What you'd get by default. Problem:
  median test-set size per class is around 8 images, several classes
  have 4. Per-class accuracy estimates have wide confidence intervals
  — an accuracy number computed from 8 test images can swing ±20
  percentage points between random subsamples. R2-Q1's Considerations
  section explicitly flags this as too noisy.

- **Per-host group** (13 PD hosts). Collapses across diseases of the
  same plant. Larger test-set sizes per group (typically 20–80
  images), more reliable accuracy estimates, but you lose the disease-
  specific signal — you can't say "the classifier transferred for
  rust but failed for blight" at this level.

- **Per-disease-category** (4–5 categories: fungal, bacterial, viral,
  pest, healthy). Collapses across hosts of the same disease type.
  Smallest number of bins, largest test-set per bin, but the bins are
  broad enough that within-bin heterogeneity dominates.

- **Custom grouping** that you can defend. For example, a hand-
  curated set of "transfer-meaningful" host-disease pairs where both
  datasets have enough images for a stable estimate.

Pick one. Write your choice and your reasoning into the JSON template
at the bottom of this section. The reasoning is what makes it a
pre-commit rather than a tag — you should be able to defend the
choice without having seen your results.

### 8.2) Decision 2 — Class-space remapping

Once you've picked an aggregation level, you need to specify how each
PV class maps into that level — and how each PD class maps into the
same level. The mapping is what defines "the same class" across the
two datasets.

The starter remapping in the next cell is one defensible version,
keyed by PV's `class_label` string with values at the host-group
aggregation level. **Treat it as a draft to push against, not as
authoritative.** It exists to make the structure of the decision
visible: you're committing to a function from PV class labels to
aggregated labels, and a parallel function from PD class labels to
the same aggregated labels.

Reasonable changes to make:

- **Different aggregation level.** If you picked disease-category
  instead of host-group, the dict values should be category strings,
  not host strings.
- **Different host alignment.** The starter treats "Corn_(maize)" (PV)
  and "Corn" (PD) as the same host group "Corn"; you might prefer to
  drop the parenthetical entirely, or to use a different canonical
  spelling.
- **Dropping classes.** The starter keeps every class; you might
  decide the orphan PD class (zero test images), or PV-only hosts
  (Blueberry, Raspberry, Soybean), shouldn't appear in the remapping
  at all.

What matters is that the remapping be explicit, complete (no UNMAPPED
output when you apply it), and committed in writing before you train.

In [ ]:
### 8.3) Starter PV → PD class-space remapping
# Starter mapping at the host-group aggregation level.
# Keys are PV `class_label` strings (the host___disease folder-name format).
# Values are host group names that should align with PD's `host` column.
# Edit this dict to match your own aggregation level and your own
# remapping decisions before saving your precommit.json.
pv_to_aggregated = {
    "Apple___Apple_scab":                                 "Apple",
    "Apple___Black_rot":                                  "Apple",
    "Apple___Cedar_apple_rust":                           "Apple",
    "Apple___healthy":                                    "Apple",
    "Blueberry___healthy":                                "Blueberry",      # PV-only
    "Cherry_(including_sour)___Powdery_mildew":           "Cherry",         # PV-only
    "Cherry_(including_sour)___healthy":                  "Cherry",         # PV-only
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "Corn",
    "Corn_(maize)___Common_rust_":                        "Corn",
    "Corn_(maize)___Northern_Leaf_Blight":                "Corn",
    "Corn_(maize)___healthy":                             "Corn",
    "Grape___Black_rot":                                  "Grape",
    "Grape___Esca_(Black_Measles)":                       "Grape",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)":         "Grape",
    "Grape___healthy":                                    "Grape",
    "Orange___Haunglongbing_(Citrus_greening)":           "Orange",         # PV-only
    "Peach___Bacterial_spot":                             "Peach",          # PV-only
    "Peach___healthy":                                    "Peach",          # PV-only
    "Pepper,_bell___Bacterial_spot":                      "Bell pepper",    # PD spells it "Bell pepper"
    "Pepper,_bell___healthy":                             "Bell pepper",
    "Potato___Early_blight":                              "Potato",
    "Potato___Late_blight":                               "Potato",
    "Potato___healthy":                                   "Potato",
    "Raspberry___healthy":                                "Raspberry",      # PV-only
    "Soybean___healthy":                                  "Soybean",        # PD spells it "Soyabean"
    "Squash___Powdery_mildew":                            "Squash",         # PV-only
    "Strawberry___Leaf_scorch":                           "Strawberry",
    "Strawberry___healthy":                               "Strawberry",
    "Tomato___Bacterial_spot":                            "Tomato",
    "Tomato___Early_blight":                              "Tomato",
    "Tomato___Late_blight":                               "Tomato",
    "Tomato___Leaf_Mold":                                 "Tomato",
    "Tomato___Septoria_leaf_spot":                        "Tomato",
    "Tomato___Spider_mites Two-spotted_spider_mite":      "Tomato",
    "Tomato___Target_Spot":                               "Tomato",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":             "Tomato",
    "Tomato___Tomato_mosaic_virus":                       "Tomato",
    "Tomato___healthy":                                   "Tomato",
}

# PD's host column is already at this aggregation level — so the PD
# side of the remapping is just metadata['host'] directly (modulo the
# Soybean/Soyabean and Corn/Corn_(maize) spelling differences you saw
# in Section 4).
print(f"PV classes mapped:        {len(pv_to_aggregated)}")
print(f"Unique aggregated groups: {sorted(set(pv_to_aggregated.values()))}")
print(f"PD unique hosts:          {sorted(metadata['host'].unique())}")

### 8.4) Decision 3 — Statistical test

Your hypothesis requires that "PV-internal accuracy" and "PV → PD
accuracy" be *statistically distinguishable*, but doesn't specify how
that's measured. You need to commit to one approach before measuring.

Reasonable options:

- **Confidence intervals on each accuracy.** Compute a 95% confidence
  interval (e.g., via Clopper-Pearson for a binomial proportion) for
  each of the two accuracy numbers. If the intervals don't overlap,
  the difference is significant at roughly the 0.05 level.

- **Bootstrap comparison of the two distributions.** Resample test
  predictions with replacement, compute each accuracy on each
  bootstrap sample, and report a 95% interval on the *difference*. If
  the interval excludes zero, the gap is real at the 0.05 level. This
  is the most flexible option and the easiest to defend across
  different aggregation levels.

- **Formal hypothesis test of difference.** McNemar's test if you
  could pair predictions on the same set of test images (you can't
  here — PV and PD are different images), or a two-proportion z-test
  treating PV-internal and PV→PD as independent samples. Cleaner but
  less flexible than bootstrap.

If you don't have a strong prior on which to use, the bootstrap
comparison is the conservative default — it doesn't require
independence or parametric assumptions, and it works the same way
regardless of which aggregation level you picked.

Whichever you pick: commit in writing before running the comparison.

### 8.5) How to save your precommit.json

Below is a template for `precommit.json`. Copy it into a new text
file, fill in your three decisions and your reasoning for each, and
save it to your R2-Q1 output directory (the path `iri.setup()`
printed at the top of this notebook). The file is your EQ#1
deliverable for Week 1.

```json
{
  "student_id": "<your iRI student id>",
  "rationale_question": "R2-Q1",
  "submitted_at": "<ISO 8601 date, e.g. 2026-06-26>",

  "decisions": {
    "aggregation_level": {
      "choice": "<one of: per_class | per_host | per_disease_category | custom>",
      "rationale": "<2–4 sentences explaining why this level fits your question>"
    },

    "class_space_remapping": {
      "description": "<short prose summary of your remapping approach>",
      "pv_to_aggregated": {
        "<pv class_label>": "<aggregated label>",
        "...": "..."
      },
      "pd_to_aggregated": {
        "<pd class_label>": "<aggregated label>",
        "...": "..."
      },
      "dropped_classes": [
        "<list any classes you're excluding, with one-sentence reasons>"
      ],
      "rationale": "<2–4 sentences explaining the remapping choices that aren't obvious>"
    },

    "statistical_test": {
      "choice": "<one of: confidence_intervals | bootstrap_difference | two_proportion_z | mcnemar | other>",
      "rationale": "<2–4 sentences explaining the choice>",
      "significance_threshold": 0.05
    }
  }
}
```

Two practical notes. First, the `pv_to_aggregated` and
`pd_to_aggregated` dicts should be complete — every class in each
dataset should appear as a key, even if its value is `"DROPPED"`. The
goal is for a reader to be able to reconstruct your evaluation by
reading the JSON alone. Second, save the file with the literal name
`precommit.json` (not `precommit_v2.json` or
`gerald_precommit.json`) because the Week 2 notebook will look for it
by that exact name.

## 9) What's next

You've now seen PlantDoc's structure (28 classes, 13 hosts, lowercased
disease vocabulary), its irregularities (inconsistent naming, the
orphan class), and its visual character (heterogeneous field-condition
photographs). You've also compared it directly with PlantVillage and
committed to the three methodology decisions that make your transfer
evaluation rigorous.

Notebook 03 (`03_baseline_classifier.ipynb`) trains a baseline CNN
classifier on PlantVillage. It expects your `precommit.json` to exist
in your R2-Q1 output directory — Week 2 starts with reading the file,
applying your remapping, and only then beginning to train. Don't move
on to Notebook 03 until your pre-commits are saved.

This notebook produced one saved artifact (`precommit.json`, written
by you). See you in Week 2.